# Notebook 03 — Evaluation and Analysis

## Purpose

This notebook provides a **comprehensive evaluation** of the multi-agent RAG system built in Notebook 02.  
It goes beyond basic retrieval metrics and covers:

| Analysis type | What is measured |
|---|---|
| Quantitative metrics | Standard IR metrics: P@k, Recall@k, MRR, NDCG |
| Statistical significance | Paired t-tests between strategy pairs |
| Agent complementarity | How much unique value each retriever contributes |
| Explainability | Why the orchestrator chose a particular strategy |
| Failure analysis | Which queries fail and why |
| Efficiency | Latency, P95 latency, total compute |
| Adaptive orchestration (Bonus) | Q-learning to improve strategy selection over time |
| Adversarial queries (Bonus) | System robustness against noisy/ambiguous inputs |
| Human-in-the-loop (Bonus) | Simulated feedback loop for post-hoc correction |

## Prerequisites

- You must have completed **Notebook 02** or have the three orchestration functions (`waterfall_orchestrate`, `voting_orchestrate`, `confidence_orchestrate`) and their wrapper classes in scope.
- The same file paths from Notebook 02 apply here (`PATH_BM25_PICKLE`, `PATH_QA`, `PATH_QRELS_FIXED`, etc.).
- All retrievers must be loaded (`bm25_fixed_qe`, `dense_fixed`, `graph_rag`).

## Section 1 — Installation and Imports

Additional dependencies beyond Notebook 02:
- **`scipy`** — paired t-tests for statistical significance testing
- **`seaborn` / `plotly`** — metric distribution plots and latency bar charts
- **`scikit-learn`** — confusion matrices and clustering utilities

In [ ]:
%pip install -q pytrec_eval scikit-learn seaborn plotly scipy

In [ ]:
# Standard library
import pickle
import json
import time
import re
import pathlib
import warnings
import functools
import importlib.machinery
from collections import defaultdict, Counter
from typing import List, Dict, Optional, Tuple, Literal
warnings.filterwarnings("ignore")

# Scientific computing
import numpy as np
import pandas as pd
from scipy import stats
from tqdm import tqdm

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# PyTorch
import torch

# NLP utilities
import nltk
from langdetect import detect
from rank_bm25 import BM25Okapi

# Hugging Face Transformers
from transformers import (
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForSeq2SeqLM, AutoTokenizer,
)

# LangChain
from langchain_core.documents import Document

# Evaluation
import pytrec_eval

# NLTK
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)
STOP_EN = set(nltk.corpus.stopwords.words("english"))
STOP_DE = set(nltk.corpus.stopwords.words("german"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Section 2 — Configuration

Same paths as Notebook 02. Edit the `ROOT` variable to match your local layout.

In [ ]:
ROOT = pathlib.Path("/path/to/your/data/root").resolve()

PATH_BM25_PICKLE  = ROOT / "storage/subsample/retrieval_downstream/bm25_fixed_qe.pkl"
PATH_DENSE_LOADER = ROOT / "storage/subsample/vectordb_dense/load_dense_fixed.py"
PATH_GRAG_LOADER  = ROOT / "storage/subsample/retrieval_graph/load_graphrag.py"
PATH_QA           = ROOT / "benchmark/benchmark_qa_bilingual.json"
PATH_QRELS_FIXED  = ROOT / "benchmark/score/fixed_size"

# pytrec_eval metric identifiers
METRICS = {"P_5", "P_10", "recall_100", "recip_rank", "ndcg_cut_10"}

## Section 3 — Re-load Step 2 Components

This section re-instantiates all components from Notebook 02.  
If you are running this notebook in the same kernel session where Notebook 02 already ran, you can skip the loading cells — the variables are already in memory.

If you are starting fresh, run all cells here before proceeding to the evaluation sections.

In [ ]:
# ── Translator ────────────────────────────────────────────────────────────────
class EnDeTranslator:
    def __init__(self, model: str = "facebook/m2m100_418M", device: str | None = None) -> None:
        self.device = device or DEVICE
        self.tok = M2M100Tokenizer.from_pretrained(model)
        self.model = M2M100ForConditionalGeneration.from_pretrained(model).to(self.device)

    @functools.lru_cache(maxsize=512)
    def translate(self, text: str, tgt: str) -> str:
        src = detect(text) if text.strip() else "en"
        src = src if src in ("en", "de") else "en"
        if src == tgt:
            return text
        self.tok.src_lang = src
        ids = self.tok(text, return_tensors="pt").to(self.device)
        out = self.model.generate(**ids, forced_bos_token_id=self.tok.get_lang_id(tgt),
                                   num_beams=1, max_new_tokens=128)
        return self.tok.decode(out[0], skip_special_tokens=True)

translator = EnDeTranslator()


# ── BilingualBM25 + QEBM25 ────────────────────────────────────────────────────
class BilingualBM25:
    def __init__(self, docs: List[Document]) -> None:
        self.docs_by_lang = {"en": [], "de": []}
        self.toks_by_lang = {"en": [], "de": []}
        for d in docs:
            lang = d.metadata.get("language", "en")
            lang = lang if lang in ("en", "de") else "en"
            self.docs_by_lang[lang].append(d)
            self.toks_by_lang[lang].append(nltk.word_tokenize(d.page_content))
        self.bm25 = {l: BM25Okapi(tok) for l, tok in self.toks_by_lang.items() if tok}

    def _rank_lang(self, q: str, lang: str, k: int) -> List[Document]:
        scores = self.bm25[lang].get_scores(nltk.word_tokenize(q))
        idx = np.argsort(scores)[::-1][:k]
        hits = []
        for i in idx:
            d = self.docs_by_lang[lang][i]
            d.metadata["bm25_score"] = float(scores[i])
            hits.append(d)
        return hits

    def search(self, query: str, top_k: int = 100) -> List[Document]:
        src = detect(query) if query.strip() else "en"
        src = src if src in ("en", "de") else "en"
        bag = []
        for lang in ("en", "de"):
            q_lang = translator.translate(query, lang) if lang != src else query
            bag.extend(self._rank_lang(q_lang, lang, top_k))
        best: Dict[str, Document] = {}
        for d in bag:
            uid = d.metadata.get("chunk_id") or d.metadata.get("record_id")
            if uid not in best or d.metadata["bm25_score"] > best[uid].metadata["bm25_score"]:
                best[uid] = d
        return sorted(best.values(), key=lambda d: d.metadata["bm25_score"], reverse=True)[:top_k]


def _expand_query(query: str, base_retriever, fb_docs: int = 5, fb_terms: int = 5) -> str:
    hits = base_retriever.search(query, top_k=fb_docs)
    tokens = [t.lower() for h in hits for t in nltk.word_tokenize(h.page_content.lower())
              if t.isalpha() and t not in STOP_EN and t not in STOP_DE]
    extra = " ".join(w for w, _ in nltk.FreqDist(tokens).most_common(fb_terms))
    return f"{query} {extra}" if extra else query


class QEBM25:
    def __init__(self, base: BilingualBM25) -> None:
        self.base = base
    def search(self, query: str, top_k: int = 100) -> List[Document]:
        return self.base.search(_expand_query(query, self.base), top_k)


print("✓ BM25 classes defined")

In [ ]:
# ── Load retrievers ────────────────────────────────────────────────────────────
with open(PATH_BM25_PICKLE, "rb") as fh:
    bm25_fixed_qe = pickle.load(fh)
bm25_fixed_qe.base.translator = translator
print("✓ BM25 loaded")

dense_loader = importlib.machinery.SourceFileLoader("dense_mod", str(PATH_DENSE_LOADER)).load_module()
dense_fixed  = dense_loader.load_dense_fixed(device=DEVICE, k=100)
print("✓ Dense loaded")

grag_loader = importlib.machinery.SourceFileLoader("grag_mod", str(PATH_GRAG_LOADER)).load_module()
graph_rag   = grag_loader
print("✓ GraphRAG loaded")

In [ ]:
# ── Query Classifier ──────────────────────────────────────────────────────────
class QueryClassifierAgent:
    def __init__(self, model_name: str = "google/flan-t5-base", max_new_tokens: int = 32, device: str = DEVICE):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
        self.max_new_tokens = max_new_tokens

    def classify(self, query: str) -> str:
        prompt = ("Classify as FACTOID, SEMANTIC, or HYBRID.\n\n"
                  "FACTOID = names, numbers, dates, who/when/where\n"
                  "SEMANTIC = why, how, explain, conceptual\n"
                  "HYBRID = mix of both\n\nQuery: " + query + "\n\nAnswer:")
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens, do_sample=False)
        return self.tokenizer.decode(out[0], skip_special_tokens=True).strip().upper()

classifier = QueryClassifierAgent()
print("✓ Classifier loaded")

In [ ]:
# ── RRF fusion utilities ──────────────────────────────────────────────────────
def _uid(doc): return doc.metadata.get("chunk_id") or doc.metadata.get("record_id")

def _safe_unique(docs):
    out, seen = [], set()
    for d in docs:
        u = _uid(d)
        if u is None or u in seen: continue
        seen.add(u); out.append(d)
    return out

def _rrf_fuse(runs, k_rrf=60, weights=None):
    weights = weights or {"bm25": 1.0, "dense": 1.0, "graph": 0.7}
    scores, store = defaultdict(float), {}
    for name, docs in runs.items():
        w = float(weights.get(name, 1.0))
        for rank, d in enumerate(docs, start=1):
            u = _uid(d)
            if u is None: continue
            store.setdefault(u, d)
            scores[u] += w / (k_rrf + rank)
    fused = sorted(store.values(), key=lambda d: scores[_uid(d)], reverse=True)
    for d in fused: d.metadata["fused_score"] = float(scores[_uid(d)])
    return fused

def orchestrate_parallel_fusion(query, top_k=5, pre_k=None, use_graph=True, weights=None, apply_overlap_rerank=False):
    trace, pre_k = [], pre_k or max(30, top_k * 10)
    bm25_docs  = _safe_unique(bm25_fixed_qe.search(query, top_k=pre_k))
    dense_docs = _safe_unique(dense_fixed.search(query, top_k=pre_k))
    graph_docs = _safe_unique(graph_rag.retrieve(query, top_k=pre_k)) if use_graph else []
    trace += [f"BM25={len(bm25_docs)}", f"Dense={len(dense_docs)}", f"Graph={len(graph_docs)}"]
    fused = _rrf_fuse({"bm25": bm25_docs, "dense": dense_docs, "graph": graph_docs},
                      weights=weights or {"bm25": 1.2, "dense": 1.0, "graph": 0.6})
    return fused[:top_k], trace

def waterfall_orchestrate(query, top_k=5):
    docs, trace = orchestrate_parallel_fusion(query, top_k, use_graph=False, weights={"bm25": 1.2, "dense": 1.0, "graph": 0.0})
    q_terms  = set(t.lower() for t in query.split() if t.strip())
    top_text = (docs[0].metadata.get("original_text") or docs[0].page_content or "").lower() if docs else ""
    overlap  = len(q_terms & set(top_text.split())) / max(len(q_terms), 1)
    if overlap < 0.05:
        docs2, trace2 = orchestrate_parallel_fusion(query, top_k, use_graph=True, weights={"bm25": 1.1, "dense": 1.0, "graph": 0.7})
        return docs2, trace + [f"fallback(overlap={overlap:.3f})"] + trace2
    return docs, trace + [f"ok(overlap={overlap:.3f})"]

def voting_orchestrate(query, top_k=5):
    return orchestrate_parallel_fusion(query, top_k, use_graph=True, weights={"bm25": 1.2, "dense": 1.0, "graph": 0.6})

def confidence_orchestrate(query, top_k=5):
    qc = classifier.classify(query)
    if qc == "FACTOID":  w, mode = {"bm25": 1.4, "dense": 0.9, "graph": 0.5}, f"factoid"
    elif qc == "SEMANTIC": w, mode = {"bm25": 0.9, "dense": 1.3, "graph": 0.6}, f"semantic"
    else:                w, mode = {"bm25": 1.0, "dense": 1.1, "graph": 0.5}, f"hybrid"
    docs, trace = orchestrate_parallel_fusion(query, top_k, use_graph=True, weights=w)
    return docs, [f"router:{mode}"] + trace

class WaterfallRetriever:
    name = "Waterfall"
    def search(self, query, top_k=100): docs, _ = waterfall_orchestrate(query, top_k); return docs

class VotingRetriever:
    name = "Voting"
    def search(self, query, top_k=100): docs, _ = voting_orchestrate(query, top_k); return docs

class ConfidenceRetriever:
    name = "Confidence"
    def search(self, query, top_k=100): docs, _ = confidence_orchestrate(query, top_k); return docs

print("✓ Orchestration functions and wrappers ready")

## Section 4 — Load Data and Relevance Judgements

### QA Benchmark

25 bilingual question-answer pairs evaluated using both English and German query variants.

### Qrels (Relevance Judgements)

Ground-truth relevance is stored as one JSON file per document in `PATH_QRELS_FIXED`.  
Each file has the form `{query_id: {"relevance_score": float}}`.  
Documents with `relevance_score >= 0.5` are considered **relevant** (binary judgement).

In [ ]:
# Load QA data
with open(PATH_QA, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

# Load relevance judgements
def load_qrels(folder: pathlib.Path) -> dict:
    qrels = defaultdict(dict)
    for fp in folder.glob("*.json"):
        did = fp.stem
        for qid, pay in json.loads(fp.read_text()).items():
            if pay["relevance_score"] >= 0.5:
                qrels[qid][did] = 1
    return qrels

QRELS = load_qrels(PATH_QRELS_FIXED)

print(f"✓ QA pairs loaded     : {len(qa_data)}")
print(f"✓ Qrels loaded        : {len(QRELS)} queries with relevance judgements")
print(f"  Example: QID=1 → {len(QRELS.get('1', {}))} relevant documents")

## Section 5 — Comprehensive Evaluation Framework

The `ComprehensiveEvaluator` class centralises all evaluation logic:

1. **`evaluate_retriever()`** — run a single strategy over all QA pairs, collect metrics + timing
2. **`compare_strategies()`** — produce a comparison table across all evaluated strategies
3. **`plot_metric_distributions()`** — violin/box plots showing per-query metric distributions
4. **`plot_latency_comparison()`** — bar chart of average and P95 latency per strategy
5. **`statistical_significance_test()`** — paired t-test comparing two strategies on a given metric

### Why per-query results matter

Macro-averaged metrics can hide large variance. A strategy with mean MRR=0.7 could be excellent on 90% of queries but completely fail on 10%. The distribution plots reveal this.

In [ ]:
class ComprehensiveEvaluator:
    """
    Evaluation framework for multi-agent RAG orchestration strategies.

    Collects per-query TREC metrics and latency for each strategy and
    provides comparison, visualisation, and statistical testing utilities.
    """

    def __init__(self, qrels: dict, metrics: set = None):
        self.qrels   = qrels
        self.metrics = metrics or METRICS
        self.results: Dict[str, dict] = {}

    def evaluate_retriever(self, retriever, qa_data: list, name: str) -> dict:
        """
        Run `retriever` over all queries in `qa_data`, compute TREC metrics,
        and record per-query latency.

        Returns a dict with keys:
          - 'metrics_mean'  : macro-averaged metric scores
          - 'per_query'     : per-query metric scores (for distribution plots)
          - 'efficiency'    : latency statistics
        """
        run     = defaultdict(dict)
        timings = []

        print(f"\n{'='*60}")
        print(f"Evaluating: {name}")
        print(f"{'='*60}")

        for q in tqdm(qa_data, desc=f"{name}"):
            qid = str(q["id"])
            t0  = time.time()
            docs = retriever.search(q["question"], top_k=100)
            timings.append(time.time() - t0)

            for rank, d in enumerate(docs, start=1):
                did = d.metadata.get("chunk_id") or d.metadata.get("record_id")
                if did:
                    run[qid][did] = 100 - rank

        # Compute per-query TREC metrics
        per_query_df = pd.DataFrame(
            pytrec_eval.RelevanceEvaluator(self.qrels, self.metrics).evaluate(run)
        ).T

        timings_arr = np.array(timings)
        efficiency  = {
            "avg_latency"   : float(timings_arr.mean()),
            "p95_latency"   : float(np.percentile(timings_arr, 95)),
            "total_time"    : float(timings_arr.sum()),
        }

        self.results[name] = {
            "metrics_mean" : per_query_df.mean().to_dict(),
            "per_query"    : per_query_df,
            "efficiency"   : efficiency,
        }

        print(f"  MRR={self.results[name]['metrics_mean'].get('recip_rank', 0):.3f}  "
              f"NDCG@10={self.results[name]['metrics_mean'].get('ndcg_cut_10', 0):.3f}  "
              f"avg_latency={efficiency['avg_latency']:.3f}s")
        return self.results[name]

    def compare_strategies(self) -> pd.DataFrame:
        """Return a strategy × metric comparison table (macro-averaged)."""
        rows = {name: res["metrics_mean"] for name, res in self.results.items()}
        return pd.DataFrame(rows).T.round(3)

    def plot_metric_distributions(self, metric: str):
        """Box plot of per-query `metric` values across strategies."""
        data = {
            name: res["per_query"][metric].values
            for name, res in self.results.items()
            if metric in res["per_query"].columns
        }
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.boxplot(data.values(), labels=data.keys())
        ax.set_title(f"Per-query {metric} distribution")
        ax.set_ylabel(metric)
        plt.tight_layout()
        plt.show()

    def plot_latency_comparison(self):
        """Bar chart comparing avg and P95 latency per strategy."""
        strategies = list(self.results.keys())
        avg = [self.results[s]["efficiency"]["avg_latency"] for s in strategies]
        p95 = [self.results[s]["efficiency"]["p95_latency"] for s in strategies]
        x = range(len(strategies))
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar([i - 0.2 for i in x], avg, 0.4, label="Avg latency")
        ax.bar([i + 0.2 for i in x], p95, 0.4, label="P95 latency")
        ax.set_xticks(list(x))
        ax.set_xticklabels(strategies)
        ax.set_ylabel("Seconds")
        ax.set_title("Latency comparison")
        ax.legend()
        plt.tight_layout()
        plt.show()

    def statistical_significance_test(self, name_a: str, name_b: str, metric: str) -> dict:
        """
        Paired t-test comparing strategy `name_a` vs `name_b` on `metric`.

        A paired t-test is appropriate here because each query is evaluated
        by both strategies, so the samples are dependent (paired per query).
        """
        a = self.results[name_a]["per_query"][metric]
        b = self.results[name_b]["per_query"][metric]
        # Align on shared query IDs
        shared = a.index.intersection(b.index)
        t_stat, p_val = stats.ttest_rel(a.loc[shared], b.loc[shared])
        return {
            "mean_diff"  : float((a.loc[shared] - b.loc[shared]).mean()),
            "t_statistic": float(t_stat),
            "p_value"    : float(p_val),
            "significant": p_val < 0.05,
        }

## Section 6 — Run Quantitative Evaluation

Evaluate all three orchestration strategies and the two base retrievers (BM25, Dense) over the QA benchmark.  
This cell is the most compute-intensive — it runs 5 retrieval strategies × 25 queries = 125 retrieval calls.

**Expected runtime**: 10–60 minutes depending on hardware (GPU vs CPU, with/without GraphRAG).

In [ ]:
evaluator = ComprehensiveEvaluator(QRELS)

strategies = {
    "Waterfall":  WaterfallRetriever(),
    "Voting":     VotingRetriever(),
    "Confidence": ConfidenceRetriever(),
}

for name, retriever in strategies.items():
    evaluator.evaluate_retriever(retriever, qa_data, name)

print("\n" + "="*80)
print("QUANTITATIVE EVALUATION RESULTS")
print("="*80)
display(evaluator.compare_strategies())

## Section 7 — Metric Distribution Visualisations

The box plots below show the **distribution of per-query metric values**, not just the mean.  
Key things to look for:
- **Width of the box** — high variance means the strategy is inconsistent across queries
- **Outliers (dots below the box)** — specific queries where the strategy fails completely
- **Median position** — robust central tendency (less affected by outliers than the mean)

In [ ]:
for metric in ["recip_rank", "ndcg_cut_10", "P_5"]:
    print(f"\n── {metric} ──────────────────")
    evaluator.plot_metric_distributions(metric)

print("\n── Latency ──────────────────")
evaluator.plot_latency_comparison()

## Section 8 — Statistical Significance Testing

### Why significance testing?

With only 25 query pairs, differences in macro-averaged metrics can easily arise from chance.  
A **paired t-test** tests the null hypothesis: *"the mean per-query metric difference is zero"*.

### Interpretation

- **p < 0.05** → the observed difference is statistically significant at 95% confidence
- **p ≥ 0.05** → we cannot reject the null hypothesis — the difference may be noise

### Limitation

With n=25 queries, the test has limited **statistical power** — it may miss real differences.  
A larger benchmark would improve reliability.

In [ ]:
print("STATISTICAL SIGNIFICANCE TESTS (paired t-test on recip_rank)")
print("=" * 70)

comparisons = [
    ("Waterfall", "Voting"),
    ("Waterfall", "Confidence"),
    ("Voting",    "Confidence"),
]

for s1, s2 in comparisons:
    r = evaluator.statistical_significance_test(s1, s2, "recip_rank")
    sig = "✓ SIGNIFICANT" if r["significant"] else "✗ not significant"
    print(f"\n{s1} vs {s2}:")
    print(f"  Mean diff  : {r['mean_diff']:+.4f}  (positive = {s1} is better)")
    print(f"  t-statistic: {r['t_statistic']:.4f}")
    print(f"  p-value    : {r['p_value']:.4f}  → {sig}")

## Section 9 — Agent Complementarity Analysis

### Motivation

Fusion only adds value if the three retrievers **disagree** — i.e., each finds some documents the others miss.  
If BM25, Dense, and GraphRAG return nearly identical result sets, fusion is redundant.

The `AgentComplementarityAnalyzer` quantifies overlap using **Venn diagram counts**:
- `bm25_only` — docs found only by BM25
- `dense_only` — docs found only by Dense
- `graph_only` — docs found only by GraphRAG
- `all_three`  — docs found by all three retrievers
- Pairwise intersections

High `bm25_only + dense_only + graph_only` values indicate **good complementarity** and justify using all three.

In [ ]:
class AgentComplementarityAnalyzer:
    """
    Measures how much each retriever contributes uniquely to the result set.

    For a given query, it computes the Venn diagram counts across the three
    retriever result sets (top-k documents each) and reports:
    - Unique contributions (docs only in one retriever)
    - Pairwise overlaps
    - Triple overlap (docs in all three)
    """

    def __init__(self, bm25, dense, graph_rag):
        self.bm25      = bm25
        self.dense     = dense
        self.graph_rag = graph_rag

    def analyze_overlap(self, query: str, top_k: int = 10) -> dict:
        """Compute per-retriever result sets and their overlap for one query."""
        bm25_ids  = {_uid(d) for d in self.bm25.search(query, top_k=top_k) if _uid(d)}
        dense_ids = {_uid(d) for d in self.dense.search(query, top_k=top_k) if _uid(d)}
        graph_ids = {_uid(d) for d in self.graph_rag.retrieve(query, top_k=top_k) if _uid(d)}

        all_ids = bm25_ids | dense_ids | graph_ids
        n       = max(len(all_ids), 1)

        only_bm25    = bm25_ids - dense_ids - graph_ids
        only_dense   = dense_ids - bm25_ids - graph_ids
        only_graph   = graph_ids - bm25_ids - dense_ids
        bm25_dense   = (bm25_ids & dense_ids) - graph_ids
        bm25_graph   = (bm25_ids & graph_ids) - dense_ids
        dense_graph  = (dense_ids & graph_ids) - bm25_ids
        all_three    = bm25_ids & dense_ids & graph_ids

        return {
            "query"         : query,
            "bm25_only_pct" : 100 * len(only_bm25)   / n,
            "dense_only_pct": 100 * len(only_dense)  / n,
            "graph_only_pct": 100 * len(only_graph)  / n,
            "bm25_dense_pct": 100 * len(bm25_dense)  / n,
            "bm25_graph_pct": 100 * len(bm25_graph)  / n,
            "dense_graph_pct":100 * len(dense_graph) / n,
            "all_three_pct" : 100 * len(all_three)   / n,
        }

    def batch_analyze(self, questions: List[str], top_k: int = 10) -> pd.DataFrame:
        """Analyze complementarity across a list of queries and return a DataFrame."""
        return pd.DataFrame([self.analyze_overlap(q, top_k) for q in tqdm(questions, desc="Overlap analysis")])


complementarity = AgentComplementarityAnalyzer(bm25_fixed_qe, dense_fixed, graph_rag)

# Analyse a subset of queries
test_questions = [q["question"] for q in qa_data[:15]]
batch_results  = complementarity.batch_analyze(test_questions, top_k=10)

print("\nAgent Complementarity Summary (avg % of result set):")
summary_cols = ["bm25_only_pct", "dense_only_pct", "graph_only_pct", "all_three_pct"]
print(batch_results[summary_cols].mean().round(2).to_string())

# Stacked bar chart
means = batch_results[summary_cols].mean()
fig, ax = plt.subplots(figsize=(7, 4))
means.plot(kind="bar", ax=ax)
ax.set_ylabel("% of result set")
ax.set_title("Average result-set contribution per agent")
ax.set_xticklabels([c.replace("_pct", "") for c in summary_cols], rotation=15)
plt.tight_layout()
plt.show()

## Section 10 — Explainability Analysis

### Why explainability?

A black-box RAG system that returns answers without justification is hard to debug and trust.  
The `ExplainableOrchestrator` adds **rationales** to every retrieval decision:
- Which query type was classified
- Which retrieval strategy was selected and why
- Which weights were applied to each retriever
- How many documents each retriever contributed

This is especially useful for:
1. Debugging failures ("was it classification error or retrieval error?")
2. Building trust with end users
3. Auditing system behaviour for compliance

In [ ]:
class ExplainableOrchestrator:
    """
    Wraps the confidence orchestration strategy with human-readable rationales.
    For every query it records:
    - Detected query type (FACTOID / SEMANTIC / HYBRID)
    - Selected weights and the reasoning behind them
    - Per-retriever document counts after fusion
    """

    def __init__(self, bm25, dense, graph_rag):
        self.bm25      = bm25
        self.dense     = dense
        self.graph_rag = graph_rag

    def explainable_route(self, query: str, top_k: int = 5) -> Tuple[List[Document], dict]:
        """Route the query and return (docs, explanation_dict)."""
        query_type = classifier.classify(query)

        if query_type == "FACTOID":
            weights  = {"bm25": 1.4, "dense": 0.9, "graph": 0.5}
            rationale = "Factoid queries (who/when/where) benefit from exact keyword matching → BM25 boosted."
        elif query_type == "SEMANTIC":
            weights  = {"bm25": 0.9, "dense": 1.3, "graph": 0.6}
            rationale = "Semantic queries (why/how/explain) need conceptual similarity → Dense boosted."
        else:
            weights  = {"bm25": 1.0, "dense": 1.1, "graph": 0.5}
            rationale = "Hybrid/ambiguous queries → balanced weights, slight Dense preference."

        pre_k      = max(30, top_k * 10)
        bm25_docs  = _safe_unique(self.bm25.search(query, top_k=pre_k))
        dense_docs = _safe_unique(self.dense.search(query, top_k=pre_k))
        graph_docs = _safe_unique(self.graph_rag.retrieve(query, top_k=pre_k))

        fused = _rrf_fuse({"bm25": bm25_docs, "dense": dense_docs, "graph": graph_docs}, weights=weights)

        explanation = {
            "query"       : query,
            "query_type"  : query_type,
            "rationale"   : rationale,
            "weights"     : weights,
            "counts"      : {"bm25": len(bm25_docs), "dense": len(dense_docs), "graph": len(graph_docs)},
            "top_doc_id"  : _uid(fused[0]) if fused else None,
        }
        return fused[:top_k], explanation

    @staticmethod
    def print_explanation(exp: dict):
        print(f"\n{'─'*70}")
        print(f"Query     : {exp['query']}")
        print(f"Type      : {exp['query_type']}")
        print(f"Rationale : {exp['rationale']}")
        print(f"Weights   : BM25={exp['weights']['bm25']}  Dense={exp['weights']['dense']}  Graph={exp['weights']['graph']}")
        print(f"Counts    : BM25={exp['counts']['bm25']}  Dense={exp['counts']['dense']}  Graph={exp['counts']['graph']}")
        print(f"Top doc   : {exp['top_doc_id']}")


explainable_orch = ExplainableOrchestrator(bm25_fixed_qe, dense_fixed, graph_rag)

test_queries = [
    "Who was president of ETH in 2003?",               # → FACTOID
    "What are the main research areas in climate science at ETH?",  # → SEMANTIC
    "How did ETH Zurich rank in 2020 and why?",         # → HYBRID
]

for q in test_queries:
    docs, explanation = explainable_orch.explainable_route(q, top_k=5)
    explainable_orch.print_explanation(explanation)

## Section 11 — Failure Analysis

### Identifying failures

A query is considered a **failure** if its MRR (reciprocal rank) is below a threshold (default: 0.5).  
MRR < 0.5 means the first relevant document does not appear in the top-2 results — typically indicating the system missed the relevant documents entirely.

### Failure patterns

After identifying failing queries, we analyse **why** they fail by looking at:
- **Query length distribution** — are long/short queries harder?
- **Language** — do EN queries fail more than DE queries (or vice versa)?
- **Query type** — do FACTOID queries fail more than SEMANTIC?

Understanding failure patterns guides improvements: e.g., if German queries consistently fail, the bilingual translation component needs attention.

In [ ]:
class FailureAnalyzer:
    """Identifies and categorises low-performing queries."""

    def __init__(self, qrels: dict):
        self.qrels = qrels

    def identify_failures(self, per_query_df: pd.DataFrame, threshold: float = 0.5) -> pd.DataFrame:
        """Return queries where recip_rank < threshold."""
        if "recip_rank" not in per_query_df.columns:
            return pd.DataFrame()
        return per_query_df[per_query_df["recip_rank"] < threshold].copy()

    def analyze_failure_patterns(self, failures: pd.DataFrame, qa_data: list) -> dict:
        """
        Join failing query IDs back to QA data to inspect:
        - Query text and length
        - Detected language
        - MRR score of the failure
        """
        qa_map = {str(q["id"]): q for q in qa_data}
        patterns = []
        for qid in failures.index:
            q = qa_map.get(str(qid), {})
            text = q.get("question", "?")
            try:
                lang = detect(text)
            except Exception:
                lang = "?"
            patterns.append({
                "qid"     : qid,
                "question": text,
                "lang"    : lang,
                "length"  : len(text.split()),
                "mrr"     : round(failures.loc[qid, "recip_rank"], 3),
            })
        return pd.DataFrame(patterns)


failure_analyzer = FailureAnalyzer(QRELS)

for name in strategies:
    print(f"\n{'='*70}")
    print(f"FAILURE ANALYSIS: {name}")
    print(f"{'='*70}")

    per_query = evaluator.results[name]["per_query"]
    failures  = failure_analyzer.identify_failures(per_query, threshold=0.5)

    if failures.empty:
        print("  No failures (all queries have MRR ≥ 0.5)")
    else:
        patterns = failure_analyzer.analyze_failure_patterns(failures, qa_data)
        print(f"  Total failures: {len(failures)} / {len(per_query)} queries")
        display(patterns)

## Section 12 — Final Evaluation Summary

Consolidate all findings into a single summary covering retrieval quality, latency, and key insights.

In [ ]:
print("=" * 80)
print("FINAL EVALUATION SUMMARY")
print("=" * 80)

# 1. Retrieval metrics
print("\n1. RETRIEVAL METRICS (macro-averaged over all queries):")
display(evaluator.compare_strategies())

# 2. Efficiency
print("\n2. EFFICIENCY:")
for name, res in evaluator.results.items():
    eff = res["efficiency"]
    print(f"  {name:12s}  avg={eff['avg_latency']:.3f}s  p95={eff['p95_latency']:.3f}s  total={eff['total_time']:.1f}s")

# 3. Key findings
print("\n3. KEY FINDINGS:")
best_mrr    = max(evaluator.results.items(), key=lambda x: x[1]["metrics_mean"].get("recip_rank", 0))
best_ndcg   = max(evaluator.results.items(), key=lambda x: x[1]["metrics_mean"].get("ndcg_cut_10", 0))
fastest     = min(evaluator.results.items(), key=lambda x: x[1]["efficiency"]["avg_latency"])
print(f"  Best MRR    : {best_mrr[0]}  ({best_mrr[1]['metrics_mean'].get('recip_rank', 0):.3f})")
print(f"  Best NDCG@10: {best_ndcg[0]}  ({best_ndcg[1]['metrics_mean'].get('ndcg_cut_10', 0):.3f})")
print(f"  Fastest     : {fastest[0]}  ({fastest[1]['efficiency']['avg_latency']:.3f}s avg)")

---

## BONUS SECTION — Advanced Features

The following three sections implement bonus features for advanced use cases.  
They are self-contained and do not depend on the core evaluation results above.

---

### BONUS 1 — Adaptive Orchestration with Q-Learning

#### Motivation

Static weight assignments (e.g., Voting with fixed weights) cannot improve from experience.  
The `AdaptiveOrchestrator` uses **Q-learning** — a model-free reinforcement learning algorithm — to learn which retrieval strategy works best for each query type.

#### Q-learning basics

A **Q-table** maps `(state, action) → expected reward`:
- **State** = query type (FACTOID, SEMANTIC, HYBRID)
- **Action** = weight configuration (bm25_heavy, dense_heavy, balanced, graph_heavy)
- **Reward** = retrieval success (1 if any relevant document retrieved, 0 otherwise)

Update rule:
```
Q(s, a) ← Q(s, a) + lr * (reward - Q(s, a))
```

**ε-greedy exploration**: with probability `ε`, choose a random action; otherwise choose `argmax Q(s, *)`.

Over time, Q-values converge to the expected reward per action, and the agent learns to prefer the best strategy for each query type.

In [ ]:
class AdaptiveOrchestrator:
    """
    Q-learning based orchestrator that learns optimal retrieval weights
    from experience.

    State  = query type (FACTOID | SEMANTIC | HYBRID)
    Action = one of four weight configurations
    Reward = binary retrieval success (relevant doc in top-k)
    """

    STRATEGY_WEIGHTS = {
        "bm25_heavy"  : {"bm25": 1.5, "dense": 0.8, "graph": 0.5},
        "dense_heavy" : {"bm25": 0.8, "dense": 1.5, "graph": 0.5},
        "balanced"    : {"bm25": 1.0, "dense": 1.0, "graph": 0.6},
        "graph_heavy" : {"bm25": 0.9, "dense": 0.9, "graph": 1.2},
    }

    def __init__(self, bm25, dense, graph_rag, learning_rate: float = 0.1, epsilon: float = 0.2):
        self.bm25      = bm25
        self.dense     = dense
        self.graph_rag = graph_rag
        self.lr        = learning_rate
        self.epsilon   = epsilon
        # Q-table: (query_type, strategy) → expected reward, initialised to 0.5
        self.q_table   = defaultdict(lambda: defaultdict(lambda: 0.5))
        self.strategies = list(self.STRATEGY_WEIGHTS.keys())
        self.history   = []

    def _select_strategy(self, query_type: str) -> str:
        """ε-greedy strategy selection."""
        if np.random.random() < self.epsilon:
            return np.random.choice(self.strategies)  # explore
        q_vals = self.q_table[query_type]
        return max(self.strategies, key=lambda s: q_vals[s])  # exploit

    def update_q_value(self, query_type: str, strategy: str, reward: float):
        """Bellman update (no future Q since this is a single-step bandit)."""
        old = self.q_table[query_type][strategy]
        self.q_table[query_type][strategy] = old + self.lr * (reward - old)

    def retrieve(self, query: str, top_k: int = 5):
        query_type        = classifier.classify(query)
        strategy          = self._select_strategy(query_type)
        weights           = self.STRATEGY_WEIGHTS[strategy]

        bm25_docs  = _safe_unique(self.bm25.search(query, top_k=30))
        dense_docs = _safe_unique(self.dense.search(query, top_k=30))
        graph_docs = _safe_unique(self.graph_rag.retrieve(query, top_k=30))
        fused = _rrf_fuse({"bm25": bm25_docs, "dense": dense_docs, "graph": graph_docs}, weights=weights)

        explanation = {
            "query_type"       : query_type,
            "selected_strategy": strategy,
            "weights"          : weights,
            "q_values"         : dict(self.q_table[query_type]),
        }
        return fused[:top_k], explanation


# ── Training loop ─────────────────────────────────────────────────────────────
adaptive_orch = AdaptiveOrchestrator(bm25_fixed_qe, dense_fixed, graph_rag, learning_rate=0.2, epsilon=0.3)

print("Training adaptive orchestrator...")
for q in tqdm(qa_data[:20], desc="Training"):
    docs, exp      = adaptive_orch.retrieve(q["question"], top_k=5)
    query_type     = exp["query_type"]
    strategy       = exp["selected_strategy"]

    # Reward: 1 if any retrieved doc is in the qrels for this query
    relevant_ids   = set(QRELS.get(str(q["id"]), {}).keys())
    retrieved_ids  = {_uid(d) for d in docs if _uid(d)}
    reward         = 1.0 if relevant_ids & retrieved_ids else 0.0

    adaptive_orch.update_q_value(query_type, strategy, reward)

print("\nLearned Q-values:")
for qt, qvals in adaptive_orch.q_table.items():
    best = max(qvals, key=qvals.get)
    print(f"  {qt:8s} → best strategy: {best:12s}  (Q={qvals[best]:.3f})")
    for s, v in sorted(qvals.items(), key=lambda x: -x[1]):
        print(f"              {s:12s}: {v:.3f}")

### BONUS 2 — Adversarial Query Evaluation

#### Why adversarial testing?

Real-world users do not always ask clean, well-formed questions.  
Adversarial testing probes system **robustness** by generating degraded variants of gold queries:

| Adversarial type | Transformation | Tests |
|---|---|---|
| Ambiguous | Remove years, remove named entities | Entity resolution robustness |
| Typo injection | Randomly corrupt characters | Spelling error tolerance |
| Negation | Add "not" to questions | Semantic negation handling |

A robust system should degrade gracefully — not catastrophically — when queries are imperfect.

In [ ]:
class AdversarialQueryGenerator:
    """Generate adversarial variants of benchmark queries."""

    @staticmethod
    def make_ambiguous(q: dict) -> dict | None:
        """Remove years and capitalised names to create ambiguous queries."""
        text = re.sub(r"\b(19|20)\d{2}\b", "", q["question"])
        text = re.sub(r"\b[A-Z][a-z]+\b", "", text)
        text = " ".join(text.split())
        if text and text != q["question"]:
            return {"id": f"adv_ambig_{q['id']}", "question": text,
                    "answer": q["answer"], "original": q["question"], "type": "ambiguous"}
        return None

    @staticmethod
    def inject_typos(q: dict, error_rate: float = 0.1) -> dict:
        """Randomly swap adjacent characters in a fraction of tokens."""
        tokens = q["question"].split()
        noisy  = []
        for t in tokens:
            if len(t) > 3 and np.random.random() < error_rate:
                i   = np.random.randint(0, len(t) - 1)
                lst = list(t)
                lst[i], lst[i + 1] = lst[i + 1], lst[i]
                t = "".join(lst)
            noisy.append(t)
        return {"id": f"adv_typo_{q['id']}", "question": " ".join(noisy),
                "answer": q["answer"], "original": q["question"], "type": "typo"}

    def generate_all(self, qa_data: list) -> list:
        results = []
        for q in qa_data:
            ambig = self.make_ambiguous(q)
            if ambig:
                results.append(ambig)
            results.append(self.inject_typos(q))
        return results


adv_gen = AdversarialQueryGenerator()
adversarial_queries = adv_gen.generate_all(qa_data)

print(f"Generated {len(adversarial_queries)} adversarial queries")
print("\nExamples:")
for adv in adversarial_queries[:4]:
    print(f"  [{adv['type']:10s}]")
    print(f"    Original   : {adv['original']}")
    print(f"    Adversarial: {adv['question']}")
    print()

# Evaluate robustness: measure hit rate on adversarial queries
def normalize_soft(text):
    return set(re.sub(r"[^\w\s]", " ", text.lower()).split())

print("Evaluating adversarial robustness...")
adv_hits = {"Waterfall": 0, "Voting": 0, "Confidence": 0}
adv_total = len(adversarial_queries)

for adv in tqdm(adversarial_queries, desc="Adversarial eval"):
    ans_tokens = normalize_soft(adv["answer"])
    for name, fn in [("Waterfall", waterfall_orchestrate), ("Voting", voting_orchestrate), ("Confidence", confidence_orchestrate)]:
        docs, _ = fn(adv["question"], top_k=5)
        hit = any(
            len(ans_tokens & normalize_soft(d.metadata.get("original_text") or d.page_content)) >= 2
            for d in docs
        )
        adv_hits[name] += int(hit)

print("\nAdversarial Hit Rates:")
for name, hits in adv_hits.items():
    print(f"  {name:12s}: {hits / adv_total:.3f}")

### BONUS 3 — Human-in-the-Loop Simulation

#### Motivation

Fully automated RAG pipelines can fail on ambiguous or incomplete queries.  
A **human-in-the-loop** system allows a reviewer to inspect the top results and:
- Confirm that the retrieved documents are correct
- Flag incorrect or irrelevant results
- Override the generated answer

In this simulation, "human" feedback is approximated by checking whether the gold answer appears in the retrieved context (the same soft overlap criterion used for hit rate).  
This simulates a reviewer who accepts results when the answer is present and rejects when it is not.

#### Feedback loop

When feedback is negative (answer not found), the system automatically retries with a different strategy.  
This mimics a real-world workflow where a human flags a bad result and the system escalates to a more thorough (but slower) retrieval approach.

In [ ]:
class HumanInTheLoopSimulator:
    """
    Simulates a human reviewer who checks RAG output quality.

    Feedback is simulated by checking whether the gold answer tokens
    appear in the retrieved context — a proxy for human acceptance.
    On negative feedback, the system retries with a fallback strategy.
    """

    def __init__(self, primary_fn, fallback_fn, feedback_threshold: float = 0.5):
        self.primary_fn   = primary_fn    # first-pass orchestration function
        self.fallback_fn  = fallback_fn   # fallback when human rejects
        self.threshold    = feedback_threshold
        self.feedback_log = []

    def _simulate_feedback(self, docs: list, gold_answer: str) -> float:
        """Return a feedback score in [0, 1] based on answer coverage."""
        ans_tokens = normalize_soft(gold_answer)
        max_overlap = 0.0
        for d in docs:
            doc_tokens = normalize_soft(d.metadata.get("original_text") or d.page_content)
            overlap    = len(ans_tokens & doc_tokens) / max(len(ans_tokens), 1)
            max_overlap = max(max_overlap, overlap)
        return max_overlap

    def run(self, query: str, gold_answer: str, top_k: int = 5) -> dict:
        """Run primary strategy, simulate feedback, retry with fallback if needed."""
        docs, trace = self.primary_fn(query, top_k=top_k)
        score       = self._simulate_feedback(docs, gold_answer)
        accepted    = score >= self.threshold

        if not accepted:
            # Human rejected → retry with fallback strategy
            docs, trace2 = self.fallback_fn(query, top_k=top_k)
            score2       = self._simulate_feedback(docs, gold_answer)
            trace        = ["[PRIMARY REJECTED]"] + trace + ["[FALLBACK]"] + trace2
            accepted     = score2 >= self.threshold
            score        = score2

        result = {"query": query, "gold": gold_answer, "feedback_score": round(score, 3), "accepted": accepted}
        self.feedback_log.append(result)
        return result


# Run simulation: Waterfall as primary, Voting as fallback
simulator = HumanInTheLoopSimulator(
    primary_fn=waterfall_orchestrate,
    fallback_fn=voting_orchestrate,
    feedback_threshold=0.3,
)

print("Running human-in-the-loop simulation...")
for q in tqdm(qa_data[:10], desc="HITL simulation"):
    result = simulator.run(q["question"], q["answer"])

log_df = pd.DataFrame(simulator.feedback_log)
accepted_rate = log_df["accepted"].mean()
avg_score     = log_df["feedback_score"].mean()

print(f"\nHuman-in-the-Loop Results (n={len(log_df)} queries):")
print(f"  Acceptance rate : {accepted_rate:.1%}")
print(f"  Avg feedback    : {avg_score:.3f}")
display(log_df[["query", "feedback_score", "accepted"]].head(10))

## Final Summary

This notebook covered the full evaluation stack for the multi-agent RAG system:

**Core evaluation**
- Quantitative IR metrics (P@5, P@10, MRR, NDCG@10, Recall@100)
- Per-query distribution plots to identify variance and outliers
- Statistical significance tests to validate metric differences
- Efficiency analysis (latency and P95)

**Qualitative and diagnostic tools**
- Agent complementarity: confirms each retriever contributes unique documents
- Explainability: human-readable rationales for strategy selection decisions
- Failure analysis: identifies which queries fail and common failure patterns

**Advanced / bonus features**
- Adaptive orchestration with Q-learning: improves strategy selection over time
- Adversarial query evaluation: tests robustness to noisy/ambiguous inputs
- Human-in-the-loop simulation: shows how human feedback can improve final answers

For production deployment, the most impactful improvements based on this evaluation are typically:
1. Growing the benchmark (25 queries has low statistical power)
2. Improving German query handling (often the biggest source of failures)
3. Fine-tuning the overlap threshold in the Waterfall fallback trigger